In [13]:
from pathlib import Path

path = Path("data/osm/california-260203.osm.pbf/").resolve()

print(path)

C:\Users\ddavi\Projects\scenic-route\processing\src\data\osm\california-260203.osm.pbf


In [14]:
from collections import defaultdict

import osmium
from osmium import osm
import h3


class ScenicHandler(osmium.SimpleHandler):
    def __init__(self, resolution=8):
        super().__init__()
        self.resolution = resolution
        self.cells = defaultdict(
            lambda: {
                "water": 0,
                "landcover": 0,
                "relief": 0,
                "recreation": 0,
                "viewpoint": 0,
                "urban": 0,
            }
        )  # h3_cell_id -> dict of counts

    def _get_cell(self, lat, lng):
        # Get the H3 cell for the given lat/lng
        cell = h3.latlng_to_cell(lat, lng, self.resolution)

        return self.cells[cell]

    def _score_cell(self, tags, lat, lng):
        match tags.get("tourism"):
            case "viewpoint":
                self._get_cell(lat, lng)["viewpoint"] += 1

        match tags.get("natural"):
            case "peak":
                self._get_cell(lat, lng)["viewpoint"] += 1
            case _:
                self._get_cell(lat, lng)["landcover"] += 1

        match tags.get("landuse"):
            case "forest":
                self._get_cell(lat, lng)["landcover"] += 1
            case _:
                self._get_cell(lat, lng)["urban"] += 1

        if tags.get("landcover") == "trees":
            self._get_cell(lat, lng)["landcover"] += 1

        if tags.get("waterway"):
            self._get_cell(lat, lng)["water"] += 1

        if tags.get("geological"):
            self._get_cell(lat, lng)["relief"] += 1

        match tags.get("leisure"):
            case "park":
                self._get_cell(lat, lng)["recreation"] += 1
            case "garden":
                self._get_cell(lat, lng)["recreation"] += 1

    def node(self, n: osm.Node):
        if not n.location.valid():
            return
        lat, lng = n.location.lat, n.location.lon
        tags = n.tags

        self._score_cell(tags, lat, lng)

    def way(self, w: osm.Way):
        tags = w.tags

        for n in w.nodes:
            if not n.location.valid():
                continue
            lat, lng = n.location.lat, n.location.lon

            self._score_cell(tags, lat, lng)

### Runtimes
Sparse index + filters: 4m 27.3s  
Filters: 3m 53.8s  
Handlerv1 (355664 cells): 5m 5.0s  
Handlerv2 (355581 cells): 10m 35.6s

In [15]:
from osmium.filter import TagFilter, KeyFilter

import time

start = time.time()
RESOLUTION = 8
handler = ScenicHandler(resolution=RESOLUTION)
handler.apply_file(
    path,
    locations=True,
    # idx="sparse_file_array",
    filters=[
        KeyFilter("natural", "landcover", "waterway", "tourism", "landuse", "leisure")
    ],
)

elapsed = time.time() - start
print(f"Took {elapsed:.2f}s")

print(f"Parsed {len(handler.cells)} H3 cells")

Parsed 355581 H3 cells


### Runtime: 

In [16]:
import json

with open("data/output/scenic_cells.json", "w") as f:
    json.dump(handler.cells, f)

print(f"Saved {len(handler.cells)} H3 cells")

Saved 355581 H3 cells


In [17]:
import pandas as pd
import json
import os
import numpy as np

print(os.getcwd())

c:\Users\ddavi\Projects\scenic-route\processing\src


In [18]:
# with open("data/output/scenic_cells_v1.json", "r") as f:
#     cells = json.load(f)

cells = handler.cells

In [19]:
# Convert to DataFrame for easy scoring
df = pd.DataFrame.from_dict(cells, orient="index")
df.index.name = "h3_cell"
df.reset_index(inplace=True)

# Scenic score formula
df["diversity"] = (df[["water", "landcover", "relief", "recreation", "viewpoint"]] > 0).sum(axis=1)

feature_cols = ["water", "landcover", "relief", "recreation", "viewpoint", "urban"]
df[feature_cols] = df[feature_cols].apply(np.log1p)
 
df["raw_score"] = (
    3 * df["water"]
    + 2 * df["landcover"]
    + 3 * df["relief"]
    + 2 * df["recreation"]
    + 1 * df["viewpoint"]
    + 2 * df["diversity"]
    - 2 * df["urban"]
)


# Normalize to 0–100
df["score"] = (
    (df["raw_score"] - df["raw_score"].min())
    / (df["raw_score"].max() - df["raw_score"].min())
    * 100
).clip(0, 100)

df_ranked = df.sort_values("score", ascending=False)
print(df_ranked[["h3_cell", "score"]].head(200))

               h3_cell       score
259    8829a0b431fffff  100.000000
279    8829a0b435fffff   99.506506
11546  8829a0b43dfffff   94.371005
13748  8829a0b491fffff   88.956690
17443  882832162dfffff   83.246576
...                ...         ...
54055  8829ab1b6dfffff   68.290884
33813  8829a11341fffff   68.287128
13440  8829a0b2cbfffff   68.286194
37713  8828308845fffff   68.270379
5886   8828308b17fffff   68.236110

[200 rows x 2 columns]


In [20]:
from datetime import datetime
from pathlib import Path

output_dir = Path("data/output") / datetime.now().strftime("%Y%m%d")
output_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%H%M%S")

df_ranked.to_json(output_dir / f"scenic_scores_{ts}.json", orient="records")
df_ranked.to_csv(output_dir / f"scenic_scores_{ts}.csv", index=False)

In [21]:
# Detect skew
print(df["raw_score"].describe())
print()
print(df["raw_score"].quantile([0.1, 0.33, 0.5, 0.75, 0.9, 0.95, 0.99]))


count    355581.000000
mean         12.402339
std           5.956293
min          -1.465736
25%           8.828314
50%          14.101887
75%          16.745486
max          45.166321
Name: raw_score, dtype: float64

0.10     2.000000
0.33    11.193686
0.50    14.101887
0.75    16.745486
0.90    18.556091
0.95    19.612020
0.99    22.967692
Name: raw_score, dtype: float64
